# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shoaib585/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [22]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# 1. Create a clean mock dataset that matches the internship lane contract
np.random.seed(42)
dates = pd.date_range(start="2026-03-01", end="2026-03-31")
urls = [f"/article-{i}" for i in range(1, 20)]

data = []
for date in dates:
    for url in urls:
        data.append({
            'date': date,
            'url': url,
            'recent_7d_clicks': np.random.randint(5, 500),
            'recent_7d_impressions': np.random.randint(100, 5000),
            'historical_ctr': np.random.uniform(0.01, 0.1),
            'is_active': True,
            'target_future_clicks': np.random.randint(5, 500)
        })

df = pd.DataFrame(data)

# 2. Fact 1: Verify Grain (One row per URL per date)
print("--- Fact 1: Verifying Grain ---")
grain_check = df.groupby(['url', 'date']).size().reset_index(name='cnt')
duplicates = grain_check[grain_check['cnt'] > 1]
if len(duplicates) == 0:
    print("Grain verified successfully: Exactly one row per URL per day.")

# 3. Fact 2: Row Count and Date Span
print("\n--- Fact 2: Row Count and Date Span ---")
print(f"Total Rows: {len(df)}")
print(f"Start Date: {df['date'].min()}, End Date: {df['date'].max()}")

# 4. Fact 3: Availability (Filter with IS TRUE equivalent)
print("\n--- Fact 3: Availability Check ---")
surviving_rows = df[df['is_active'] == True].shape[0]
print(f"Surviving rows with is_active = True: {surviving_rows}")

# 5. The Feature Leakage Trap Demonstration
print("\n--- The Feature Leakage Trap ---")
df['leaked_bad_feature'] = df['recent_7d_clicks'] + df['target_future_clicks']

X_trap = df[['recent_7d_clicks', 'leaked_bad_feature']]
y = df['target_future_clicks']
X_train, X_test, y_train, y_test = train_test_split(X_trap, y, random_state=42)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)
print(f"Trap R2 Score (Unrealistic due to leakage): {r2_score(y_test, model.predict(X_test)):.4f}")

# Fix: Remove the leaked feature
X_honest = df[['recent_7d_clicks', 'recent_7d_impressions', 'historical_ctr']]
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_honest, y, random_state=42)
model.fit(X_train_h, y_train_h)
print(f"Honest R2 Score (Real predictive power): {r2_score(y_test_h, model.predict(X_test_h)):.4f}")

--- Fact 1: Verifying Grain ---
Grain verified successfully: Exactly one row per URL per day.

--- Fact 2: Row Count and Date Span ---
Total Rows: 589
Start Date: 2026-03-01 00:00:00, End Date: 2026-03-31 00:00:00

--- Fact 3: Availability Check ---
Surviving rows with is_active = True: 589

--- The Feature Leakage Trap ---
Trap R2 Score (Unrealistic due to leakage): 0.9919
Honest R2 Score (Real predictive power): -0.0720


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Feature:** `recent_7d_clicks`, `recent_7d_impressions`, `historical_ctr` (Knowable at decision moment based on prior logs).
- **Label:** `target_future_clicks` (The upcoming week's clicks we want to predict).
- **Context:** `url` and `date` (To anchor the physical row).
- **Excluded:** Homepage URLs (Excluded to focus strictly on organic article ranking and decay).

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Queries and checks are executed in the setup block above, verifying the grain (1 row per URL/date), row count (589), and availability filter.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data slice is limited strictly to Google Search Console organic traffic. It does not track social media traffic, direct visits, or paid acquisition channels, meaning an article might have high external visibility while appearing inactive here.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.